# Invoice Audit — Meridian Health Assurance Group

Five hospitals bill a health insurer under five separately negotiated contracts.
Some invoices are wrong. Identify them, state what each should have totalled, and
say how confident you are.

**Scale.** 4,886 invoices, 61,211 line items, five contracts totalling 57,600 tokens.
Hospital 1 is labelled and used only for development. Hospitals 2–5 are scored.

**Result.** Precision 1.000, recall 1.000 on the development set across all eighteen
error categories, with expected totals exact on 909 of 913 invoices.

---

### Design

A language model reads English. Python does arithmetic. No value a model produces is
used until the deterministic engine reproduces the invoice totals from it.

| | |
|---|---|
| Four tabular contracts | parsed by regex |
| One prose contract (hospital 2) | extracted by a language model |
| 61,211 line items repriced | deterministic engine |
| Description → service resolution | contract-derived price index |
| Model selection | measured against a gold spec built for free |

### Contents

| | |
|---|---|
| 0 | Environment and package |
| 1 | Data |
| 2 | Checks requiring no contract |
| 3 | Reading the contracts |
| 4 | Resolving descriptions to services |
| 5 | Repricing engine |
| 6 | Evaluation on hospital 1 |
| 7 | Model selection |
| 8 | Hospital 2 |
| 9 | Submission |

> **Where this notebook stands.** This is the research record: how the contracts were explored, how four models were compared, and how the price-based resolver was validated without labels. It imports `src/` and produces nothing the runtime does not. The system itself is `python app.py`; the results are in `reports/`. Two things happened after this notebook was last run: the model pinned for hospital 2 moved from gpt-4o to deepseek after a rule-by-rule check against the contract text (`evaluation/verify_hospital_2.py`, decision log item 11), and the runtime was split from the research code. Cells that call a GPU or a live model need Colab and a key; everything else runs locally.

---

## 0 · Setup

Three cells: locate the data, write the `src/` package, import everything.

The notebook builds the package it imports, so a fresh Colab session — which
downloads the notebook and nothing else — reproduces the whole project. The modules
themselves are best read as files in `src/`.

In [ ]:
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    if not os.path.exists('/content/insurance_auditing'):
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/majedzahrani3/insurance_auditing.git',
                        '/content/insurance_auditing'], check=True)
    PROJECT, DATA = '/content', '/content/insurance_auditing'
else:
    PROJECT = os.path.abspath('..')
    DATA = os.path.join(PROJECT, 'data')

for _folder in ('src', 'prompts', 'reports', 'runs'):
    os.makedirs(os.path.join(PROJECT, _folder), exist_ok=True)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
os.chdir(PROJECT)

print(f'colab {IN_COLAB}   project {PROJECT}   data {DATA}')

In [ ]:
# Version floors, not pins: Colab runs Python 3.13 and older pinned releases have
# no wheel for it, so pip falls back to compiling tokenizers from Rust and fails.
# bitsandbytes is floored at the version current transformers requires for 4-bit.
if IN_COLAB:
    !pip install -q -U 'transformers>=4.46' 'accelerate>=1.0' 'bitsandbytes>=0.46.1' 'sentence-transformers>=3.1' 'outlines>=0.1' 'lm-format-enforcer>=0.10'
    !pip install -q pypdf pdf2image pytesseract
    !apt-get -qq install -y tesseract-ocr poppler-utils > /dev/null
    print('\nRuntime > Restart session if this was the first install, then Run all.')

### Imports

Every import the notebook uses. No cell below this one imports anything.

In [ ]:
import collections
import copy
import glob
import json
import os
import re
import shutil
import sys
import subprocess
import textwrap
import warnings
from decimal import Decimal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import ask as ask_mod
from src import audit, checks, classify, compile_contract, config, contract_table
from src import contracts, data, extract, ingest, llm, markdown_tables, scoring
from src import money, output, pricing, resolver, retrieval
from evaluation import chunking, embeddings, local_model, model_comparison, spec_diff, stress_test


try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    torch, HAS_GPU = None, False

try:
    from google.colab import drive
except ImportError:
    drive = None

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
plt.rcParams.update({'figure.figsize': (10, 4), 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.dpi': 110})

HOSPITALS = [f'hospital_{i}' for i in range(1, 6)]
DEV, SCORED = 'hospital_1', HOSPITALS[1:]
TABULAR = ['hospital_1', 'hospital_3', 'hospital_4', 'hospital_5']
PROSE = 'hospital_2'

_bnb_ok, _bnb = local_model.bitsandbytes_ready()
print(f'pandas {pd.__version__}  numpy {np.__version__}  gpu {HAS_GPU}')
print(f'4-bit quantisation: {_bnb_ok}  ({_bnb})')
if HAS_GPU and not _bnb_ok:
    print('  -> the local model will load Qwen2.5-3B in fp16 instead of 7B in 4-bit')
    print("  -> for the 7B: !pip install -U 'bitsandbytes>=0.46.1', then RESTART SESSION")
print(f'development {DEV}   scored {SCORED}')

---

## 1 · Data

Four facts about this dataset change what can be built on it. Each is established
below before any of it is relied upon.

In [ ]:
sizes = []
for h in HOSPITALS:
    inv = data.load_invoices(DATA, h)
    li = data.load_line_items(DATA, h)
    sizes.append({'hospital': h, 'invoice_rows': len(inv),
                  'unique_ids': inv.invoice_id.nunique(), 'line_items': len(li),
                  'lines_per_invoice': round(len(li) / inv.invoice_id.nunique(), 1)})
sizes = pd.DataFrame(sizes)
print(f'{sizes.invoice_rows.sum():,} invoice rows, {sizes.line_items.sum():,} line items')
sizes

**`invoice_rows` exceeds `unique_ids` in every hospital.** Either rows are
duplicated, or identifiers are reused by different invoices. These have opposite
remedies, so the difference matters.

In [ ]:
inv = data.load_invoices(DATA, DEV)
li = data.load_line_items(DATA, DEV)
reused = inv[inv.duplicated('invoice_id', keep=False)].invoice_id.unique()

print(f'identifiers used more than once: {len(reused)}')
display(inv[inv.invoice_id == reused[0]][
    ['invoice_id', 'invoice_date', 'patient_id', 'plan_tier', 'invoice_total_cents']])

groups = li[li.invoice_id == reused[0]].groupby('source_seq').line_total_cents.sum()
print('line groups by line_id prefix:')
print(groups.to_string())
print(f'invoice totals stated: {list(inv.loc[inv.invoice_id == reused[0], "invoice_total_cents"])}')

Different patient, date and total under one identifier: two distinct invoices, not
a duplicated row. The `line_id` prefix partitions them, and the two groups sum
exactly to the two stated totals.

**The invoice key is the `line_id` prefix.** Everything downstream prices an invoice
*unit*. Identifier reuse is itself one of the eighteen error categories, so the later
invoice of a pair is flagged and the first treated as legitimate.

In [ ]:
labels = data.load_labels(DATA)
n, bad = len(labels), labels.is_erroneous.sum()
counts = pd.Series(collections.Counter(c for cs in labels.categories for c in cs)).sort_values()

print(f'{bad} of {n} invoices erroneous ({bad/n:.1%})')
print(f'a model flagging nothing scores {(n-bad)/n:.1%} accuracy and finds no errors')
print(f'categories per erroneous invoice: '
      f'{dict(sorted(collections.Counter(len(c) for c in labels.categories if c).items()))}')

plt.figure(figsize=(9, 5.5))
plt.barh(counts.index, counts.values, color='#b03a3a')
plt.title(f'Error categories, hospital 1 ({len(counts)} kinds across {bad} invoices)')
plt.xlabel('invoices affected')
plt.tight_layout(); plt.show()

**Errors are 6.4% of invoices**, so accuracy is uninformative and is not reported
anywhere below. Precision and recall are.

Eighteen categories, most invoices carrying two or three, none with more than twelve
examples. Multi-label, and small supports — so every table below prints support.

In [ ]:
services_h1 = list(compile_contract.compile_contract(DATA, DEV)['services'])
uses = li.description.value_counts()

print(f'contracted services : {len(services_h1)}')
print(f'distinct descriptions: {li.description.nunique()}')
print(f'aliases per service  : {li.description.nunique() / len(services_h1):.1f}')
print(f'descriptions used once: {(uses == 1).sum()}')
print()
for d in li.description.drop_duplicates().sample(8, random_state=0):
    print('   ', d)

Abbreviated (`Rtn`, `Compr`, `Ent` for Otolaryngologic), reordered, truncated and
suffixed with noise codes. 108 services under 488 descriptions.

The twelve descriptions used exactly once are separated from the rest by an order of
magnitude. That gap is used in section 4.

In [ ]:
shapes = []
for h in HOSPITALS:
    docs = sorted(glob.glob(f'{DATA}/contracts/{h}/*.md'))
    text = ''.join(open(d).read() for d in docs)
    rows = len([l for l in text.splitlines() if l.strip().startswith('|')])
    shapes.append({'hospital': h, 'documents': len(docs), 'tokens': len(text) // 4,
                   'table_rows': rows, 'format': 'tabular' if rows > 50 else 'prose'})
pd.DataFrame(shapes)

**Hospital 2 has no tables and is nine times longer than any other.** That single
row is why a language model appears in this project at all.

In [ ]:
steps = []
for h in HOSPITALS:
    inv_h = data.load_invoices(DATA, h)
    li_h = data.load_line_items(DATA, h)
    steps.append({'hospital': h,
                  'unparseable_dates': int(li_h.service_date.isna().sum()),
                  'other_nulls': int(inv_h.drop(columns=[c for c in inv_h if c.endswith('_raw')]).isna().sum().sum()
                                     + li_h.drop(columns=['service_date', 'service_date_raw']).isna().sum().sum()),
                  'quantity<=0': int((li_h.quantity <= 0).sum()),
                  'price<=0': int((li_h.unit_price_cents <= 0).sum()),
                  'orphan_lines': int((~li_h.invoice_id.isin(set(inv_h.invoice_id))).sum())})
display(pd.DataFrame(steps))

print('the unparseable values:')
for v in li[li.service_date.isna()].service_date_raw.tolist():
    print('   ', repr(v))

The dataset is structurally clean. **The only nulls anywhere are the unparseable
service dates, and those are `malformed_service_date` — an error category.**

So `dropna()` would delete six findings, `drop_duplicates()` five more, and outlier
removal would take `unit_price_mismatch`, the largest category. **No cleaning is
performed.** Representation is normalised — integer cents, parsed dates alongside
their raw text, contract-mandated sort order — but no value is altered, and money is
never averaged.

---

## 2 · Checks requiring no contract

Seven of the eighteen categories are arithmetic or calendar facts, or need only the
contract number and term. They cost nothing per hospital, never guess, and run on
all five before any contract is parsed.

In [ ]:
headers = {h: contracts.read_header(DATA, h) for h in HOSPITALS}
pd.DataFrame([{'hospital': h, **{k: v for k, v in hd.items() if k != 'documents'},
               'documents': len(hd['documents'])} for h, hd in headers.items()])

In [ ]:
units = {}
lines = {}
tier1 = {}
for h in HOSPITALS:
    inv_h = data.load_invoices(DATA, h)
    lines[h] = data.load_line_items(DATA, h)
    units[h] = data.build_invoice_units(inv_h, lines[h])
    tier1[h] = checks.run_all(units[h], lines[h], headers[h])

pd.DataFrame([{'hospital': h,
               'invoices': units[h].invoice_id.nunique(),
               'flagged': tier1[h].invoice_id.nunique(),
               'pct': round(100 * tier1[h].invoice_id.nunique() / units[h].invoice_id.nunique(), 1)}
              for h in HOSPITALS])

In [ ]:
scores_t1 = scoring.report(tier1[DEV], labels, 'tier 1 — no contract, no model')

**Precision 1.000, recall 0.534.** Thirty-one of fifty-eight errors, no false
positives, and every one of the seven targeted categories at 1.000 on both measures.

Half the errors for no contract reading and no inference cost. The eleven categories
still at zero all require knowing what a service should have cost.

In [ ]:
for cat in ['line_total_arithmetic', 'malformed_service_date',
            'service_date_out_of_window', 'duplicate_invoice_id']:
    row = tier1[DEV][tier1[DEV].category == cat].iloc[0]
    print(f'{cat:32s} {row.detail}')

Each finding carries the evidence that produced it, so a reviewer can verify any row
without rerunning anything.

---

## 3 · Reading the contracts

All five encode the same pricing model and state the same adjustment order verbatim:
bundle substitution, facility multiplier, plan-tier multiplier, premium, cumulative
volume discount, rounding half-up after each step. They differ only in presentation,
so all five compile to one representation and the engine is written once.

In [ ]:
specs = {h: compile_contract.compile_contract(DATA, h) for h in TABULAR}
pd.DataFrame([contracts.summarise(s) for s in specs.values()])

Zero warnings: no rule table names a service the rate schedule lacks.

Hospital 3 shows **120 services across 127 priced periods** — its amendment reprices
seven services from 1 January 2025 *by service date, not invoice date*, so those carry
two rates and the engine selects by treatment date. Hospital 4 has zero
non-business-day uplifts because its section 10 states `None.`

In [ ]:
amended = {n: s for n, s in specs['hospital_3']['services'].items() if len(s['rates']) > 1}
for n, s in list(amended.items())[:5]:
    before = next(r for r in s['rates'] if r['valid_to'])
    after = next(r for r in s['rates'] if r['valid_from'])
    print(f"{n[:44]:44s} {before['cents']:>7} -> {after['cents']:>7}  from {after['valid_from']}")

### 3.2 · Onboarding a provider who sends a PDF

This exercise ships Markdown and plain text. A provider signing a contract sends a
PDF, so ingestion has to cope with one — and the three possible routes are not equally
trustworthy.

| route | when | exact |
|---|---|---|
| Markdown / plain text | supplied | yes |
| PDF with an embedded text layer | most digital PDFs | yes |
| OCR | scanned paper, no text layer | **no** |

The ordering carries real weight. OCR misreads digits — 5/S, 0/O, 1/l — and a contract
is almost entirely digits. A rate read as `25260` instead of `25250` misprices every
invoice touching that service, and nothing downstream can catch it, because the wrong
number is perfectly well-formed. So OCR is a last resort, the route taken is always
reported, and an OCR-derived contract carries reduced confidence on every rate.

In [ ]:
rows = []
for path in sorted(glob.glob(f'{DATA}/contracts/*/*')):
    text, meta = ingest.read_contract_file(path)
    rows.append({'hospital': path.split('/')[-2], **meta, 'chars': len(text)})
pd.DataFrame(rows)[['hospital', 'file', 'route', 'exact', 'chars']]

> Every PDF in this dataset carries a text layer, so OCR never runs here. Note also
> that `master_services_agreement_scanned.pdf` yields **74** rate clauses against the
> Markdown's **76** — routing through it would lose two services. The task says *"read
> whichever format suits your tooling"*, and the text form is strictly better.

In [ ]:
# The OCR path, proved on a PDF that genuinely has no text layer.
from PIL import Image, ImageDraw

_img = Image.new('RGB', (1240, 320), 'white')
_d = ImageDraw.Draw(_img)
for _i, _line in enumerate(['SCHEDULE OF RATES', '',
                            'Routine Cardiac Specimen Analysis  per item supplied  GBP 233.50',
                            'Advanced Neurological Consultation per visit          GBP 141.25']):
    _d.text((40, 30 + _i * 60), _line, fill='black')
os.makedirs(f'{PROJECT}/reports/ocr_demo', exist_ok=True)
_scanned = f'{PROJECT}/reports/ocr_demo/scanned_contract.pdf'
_img.save(_scanned, 'PDF', resolution=150)

_text, _meta = ingest.read_contract_file(_scanned)
print(f"route  {_meta['route']}")
print(f"exact  {_meta['exact']}")
print(f"note   {_meta['note']}")
print()
print('digit repair, applied inside monetary amounts only:')
for _bad in ['GBP 2S2.SO per item', 'GBP 1O1.5O per hour']:
    print(f'   {_bad:24s} -> {ingest.repair_ocr_numbers(_bad)}')

> The scanned PDF is correctly identified as having no text layer and routed to OCR.
> Where the OCR toolchain is absent the failure is reported rather than silent — an
> empty contract would otherwise yield zero services and flag every invoice as
> `unknown_service`.

> Digit repair is applied **only inside monetary amounts**. Rewriting letters to digits
> in ordinary prose would corrupt service names, and a corrupted service name fails
> loudly at resolution while a corrupted rate does not fail at all.

### 3.3 · One table, whatever arrived

Five contracts, five layouts — tables, prose, three documents, a PDF — reduced to one
row per service per rate period with identical columns. **That table is the only thing
the pricing engine reads**, so a sixth provider in a sixth format changes nothing
downstream.

```
  .md / .txt ─┐
  PDF (text) ─┼─► one normalised table ─► pricing engine ─► submission.csv
  PDF (OCR)  ─┘        ▲
                       │
         regex for tables, model for prose — the route is recorded per row
```

It is also the reviewable artefact. Written to CSV it is a few hundred rows an auditor
can read against the contract itself, which a nested specification object is not.

In [ ]:
contract_tables = []
for h in TABULAR:
    _text, _prov = ingest.ingest_contract(f'{DATA}/contracts/{h}')
    _route = 'ocr' if any(p['route'] == 'ocr' for p in _prov) else 'text'
    contract_tables.append(contract_table.spec_to_table(
        specs[h], _route, ingest.ingestion_confidence(_prov)))

contracts_table = contract_table.combine(contract_tables)
contracts_table.to_csv(f'{PROJECT}/reports/contracts_normalised.csv', index=False)
print(f'{len(contracts_table)} rows · {len(contracts_table.columns)} columns · '
      f'{contracts_table.service.nunique()} distinct services')
display(contract_table.summarise(contracts_table))

In [ ]:
contracts_table[contracts_table.valid_to.notna()][
    ['hospital', 'service', 'unit_basis', 'rate_cents', 'valid_from', 'valid_to']].head(8)

> Rows carrying a `valid_to` are hospital 3's amended services — the same service
> priced differently either side of 1 January 2025, which a single-rate table could not
> represent. The engine selects by **service date**, since the amendment states that the
> invoice date is irrelevant.

> `source_route` and `extraction_confidence` travel with every row, so a rate that came
> through OCR is visibly less trustworthy than one read from a text layer, all the way
> to the submission.

### 3.1 · A scorer, built before any model runs

A model's output is text, and text is uniformly plausible whether or not it is
correct. The marking scheme therefore comes first — and its answers are free, since
the tabular contracts are already read correctly.

Validated two ways: a spec scored against itself must be perfect, and a spec broken
in seven known ways must report exactly seven.

In [ ]:
gold = specs['hospital_5']
damaged = copy.deepcopy(gold)
names = sorted(damaged['services'])

del damaged['services'][names[0]]
damaged['services']['Invented Cardiac Nonsense'] = contracts.new_service(
    'Invented Cardiac Nonsense', 'per_visit', [contracts.new_rate(12345)])
damaged['services'][names[5]]['rates'][0]['cents'] += 5
damaged['services'][names[6]]['unit_basis'] = 'per_visit'
damaged['services'][names[7]]['daily_cap'] = 99
damaged['threshold_premiums']['Invented Cardiac Nonsense'] = (5, Decimal('0.25'))
damaged['nbd_uplifts'].pop(sorted(damaged['nbd_uplifts'])[0])

perfect = spec_diff.compare_specs(gold, gold)
print(f"self-comparison: rate_exact {perfect['rate_exact']}, "
      f"hallucinated {perfect['hallucinated_rules']}")
print()
display(spec_diff.diff_report(gold, damaged))

Seven faults, seven found, each named — including a rate wrong by five cents. Money
is compared exactly; partial credit would conceal the errors being hunted.

Missed, wrong and hallucinated rules are counted separately. A missed rule leaves a
visible gap; an invented one is a confident value with no source that misprices every
invoice touching that service.

In [ ]:
h2_text = open(f'{DATA}/contracts/{PROSE}/master_services_agreement.md').read()
clauses = re.findall(r'^\d+\.\d+ In respect of.*$', h2_text, flags=re.M)
stats = chunking.chunk_stats(h2_text)

print(f'{len(h2_text)//4:,} tokens, 0 tables, {len(clauses)} rate clauses')
print({k: stats[k] for k in ['articles_kept', 'articles_dropped', 'clauses_expected',
                             'fraction_dropped', 'largest_chunk_tokens']})
print()
print(textwrap.fill(clauses[1], 96))

One hundred and twenty words carrying three usable values: the service name, `10625`,
and `per_hour_per_item`. Seventy-six of these across thirty-five articles.

Selecting rate-bearing articles by heading drops 49.2% of the document with no recall
risk and leaves the largest chunk at 1,646 tokens. Retrieval is not used for this:
all thirteen articles are required, and top-*k* similarity cannot guarantee
completeness. Section 7 measures that claim rather than asserting it.

In [ ]:
conv = {h: chunking.convention_checklist(
            ''.join(open(f).read() for f in sorted(glob.glob(f'{DATA}/contracts/{h}/*.md'))))
        for h in [DEV, PROSE]}

for key in conv[PROSE]:
    a, b = (conv[DEV][key] or ''), (conv[PROSE][key] or '')
    same = a.replace('taken', 'counted') == b.replace('taken', 'counted')
    print(('same     ' if same else 'DIFFERENT') + f'  {key}')
    if not same:
        print(textwrap.fill(f'H1: {a}', 96, subsequent_indent=' ' * 6))
        print(textwrap.fill(f'H2: {b}', 96, subsequent_indent=' ' * 6))

Rounding, adjustment order and cumulative utilisation are identical between the two
contracts. Only the day definitions differ.

Hospital 2's Service Day runs 07:00 to 06:59, and a Business Day is one that
*commences* off a weekend — so a service delivered at 03:00 on a Monday would fall in
Sunday's Service Day and attract the weekend uplift. Eight hospital 2 services carry
such an uplift.

**The line items carry dates and no times**, so the 00:00–06:59 window cannot be
identified and the two readings are indistinguishable from the data. The recorded date
is read as the commencing date, matching hospital 1. If that is wrong, only those
eight services misprice and only adjacent to weekends — a checkable prediction,
recorded as item 6 of `reports/decision_log.md`.

---

## 4 · Resolving descriptions to services

108 services are billed under 488 descriptions. The task description frames this as
the central difficulty, and lexically it is.

It is not solved lexically. A contract can produce only a small enumerable set of unit
rates — base, bundled, each multiplied by any facility and plan-tier multiplier, any
premium, any volume discount, rounded at each step. Matching a description's modal
`(unit_basis, unit_price_cents)` against that set identifies the service.

In [ ]:
price_index = resolver.derivable_prices(specs[DEV])
example = 'Standard Otolaryngologic Radiotherapy Fraction'
legal = sorted(p for (b, p), names in price_index.items() if example in names)

print(f'{example}')
print(f"  base rate {specs[DEV]['services'][example]['rates'][0]['cents']}, "
      f"discounts {specs[DEV]['volume_discounts'][example]}")
print(f'  every price it can legally be: {legal}')
print()
print(f'index covers {len(price_index)} distinct (basis, price) keys')
print(f'keys matching more than one service: {sum(1 for v in price_index.values() if len(v) > 1)}')

In [ ]:
resolutions = {h: resolver.resolve(specs[h], lines[h]) for h in TABULAR}

pd.DataFrame([{'hospital': h,
               'descriptions': len(r),
               'by_price': int((r.method == 'price_unique').sum()),
               'text_tiebreak': int((r.method == 'price_tiebreak_text').sum()),
               'unknown_service': int((r.method == 'unknown_service').sum()),
               'unresolved': int((r.method == 'unresolved').sum()),
               'services_covered': f"{r.service.nunique()}/{len(specs[h]['services'])}"}
              for h, r in resolutions.items()])

No description is left unresolved on any hospital. Text is used only to break genuine
price collisions and, below, to contradict.

### 4.1 · Detecting services absent from the contract

A description for a service the contract does not list still lands on *some* derivable
price by coincidence. Two independent signals identify that: the text does not support
the price's choice, and no repeated usage establishes the description as an alias.

In [ ]:
r = resolutions[DEV]
unknown, genuine = r[r.method == 'unknown_service'], r[r.method != 'unknown_service']

print(f'lexical score of chosen service — unknown max {unknown.score_of_chosen.max():.3f}'
      f'  vs genuine min {genuine.score_of_chosen.min():.3f}')
print(f'times used                     — unknown max {unknown.n_rows.max()}'
      f'      vs genuine min {genuine.n_rows.min()}')

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].scatter(genuine.score_of_chosen, genuine.n_rows, s=14, alpha=.5,
              label='resolved', color='#3a6ea5')
ax[0].scatter(unknown.score_of_chosen, unknown.n_rows, s=42, marker='x',
              label='unknown_service', color='#b03a3a')
ax[0].set_xlabel('text similarity to the price-chosen service')
ax[0].set_ylabel('times the description is used'); ax[0].set_yscale('log')
ax[0].legend(); ax[0].set_title('Two signals, neither sufficient alone')
ax[1].hist([genuine.score_of_chosen, unknown.score_of_chosen], bins=25, stacked=True,
           color=['#3a6ea5', '#b03a3a'], label=['resolved', 'unknown_service'])
ax[1].set_xlabel('text similarity'); ax[1].legend(); ax[1].set_title('Similarity alone overlaps')
plt.tight_layout(); plt.show()

The score ranges overlap (0.123 against 0.095) and usage alone would be arbitrary.
Requiring both recovers all twelve `unknown_service` invoices with no false positives.

This is the most data-fitted decision in the system, calibrated on twelve examples.
The margin is reported per hospital so that a collapse is visible rather than silent —
on hospital 5 it widens to 0.121 against 0.248.

### 4.2 · Independent validation of the resolver

The resolver assigns services by price and never reads the description. A sentence
encoder reads the description and never sees a price. If the two agree, that is
evidence rather than circularity — and it needs no labels, so it also applies to the
four hospitals with no ground truth.

In [ ]:
encoder = embeddings.load_encoder()

# Every description is encoded in one pass. Encoding subsets separately would fit a
# different vocabulary each time and place them in incomparable spaces.
vectors, encoder_name = embeddings.embed(r.description.tolist(), encoder)
coords, variance = embeddings.project(vectors)

is_unknown = (r.method == 'unknown_service').values
assigned = r.service.fillna('(unknown)').tolist()
cluster_labels = embeddings.cluster(vectors[~is_unknown],
                                    n_clusters=r.service.nunique())

print(f'encoder    {encoder_name}')
print(f'vectors    {vectors.shape}, PCA explains {variance.sum():.1%} in 2D')
agree = embeddings.agreement(cluster_labels, r.service[~is_unknown].tolist())
for k, v in agree.items():
    print(f'  {k:14s} {v}')
print(f'  {"silhouette":14s} '
      f'{embeddings.separation(vectors[~is_unknown], r.service[~is_unknown].tolist())}')

In [ ]:
top = r.service.value_counts().head(12).index

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].scatter(coords[:, 0], coords[:, 1], s=10, alpha=.2, color='#bbbbbb')
for svc in top:
    m = (r.service == svc).values
    ax[0].scatter(coords[m, 0], coords[m, 1], s=34, alpha=.85, label=svc[:32])
ax[0].set_title('Description embeddings, coloured by the service the PRICE assigned')
ax[0].set_xlabel(f'PC1 ({variance[0]:.1%})')
ax[0].set_ylabel(f'PC2 ({variance[1]:.1%})')
ax[0].legend(fontsize=6, loc='best')

ax[1].scatter(coords[~is_unknown, 0], coords[~is_unknown, 1], s=12, alpha=.35,
              color='#3a6ea5', label='resolved')
ax[1].scatter(coords[is_unknown, 0], coords[is_unknown, 1], s=90, marker='x',
              linewidths=2, color='#b03a3a', label='unknown_service')
ax[1].set_title('Descriptions matching no contracted service')
ax[1].set_xlabel(f'PC1 ({variance[0]:.1%})'); ax[1].legend()
plt.tight_layout(); plt.show()

Descriptions the price index assigned to the same service occupy the same region of
embedding space, without the encoder having seen a single price. Homogeneity near 0.9
means the text-only clustering recovers the price-based assignment almost exactly.

Two methods with no shared information reaching the same answer is the strongest
check available on the four hospitals that have no labels.

---

## 5 · Repricing engine

A single ordered pass over each hospital's line items. Cumulative volume discounts
depend on utilisation *prior to* each line, counted across the whole term and all
patients, so invoices are not independent and cannot be parallelised. Per-patient
daily quantities, bundle co-occurrence and exclusion windows are precomputed.

Adjustment order is taken verbatim from the contracts, with half-up rounding after
each step.

In [ ]:
priced = {h: pricing.reprice(specs[h], units[h], lines[h], resolutions[h])
          for h in TABULAR}

pd.DataFrame([{'hospital': h,
               'line_items': len(p),
               'priced': int(p.expected_line_total.notna().sum()),
               'reproduce_billed': f'{(p.expected_line_total == p.line_total_cents).sum() / p.expected_line_total.notna().sum():.2%}',
               'differ': int((p.expected_line_total.notna() & (p.expected_line_total != p.line_total_cents)).sum())}
              for h, p in priced.items()])

Between 99.30% and 99.51% of line items reproduce the billed amount exactly. The
remainder are the mispricings.

That figure is also the validation of every contract reading: a misread rate would
surface here as thousands of mismatches, not dozens.

In [ ]:
step, once = 0, 0
for name, svc in specs['hospital_5']['services'].items():
    base = svc['rates'][0]['cents']
    facs = specs['hospital_5']['facility_multipliers'].get(name, {})
    tiers = specs['hospital_5']['tier_multipliers'].get(name, {})
    for f in facs.values():
        for t in tiers.values():
            step += 1
            if money.apply(money.apply(base, f), t) != money.half_up(Decimal(base) * f * t):
                once += 1
print(f'hospital 5: {once} of {step} rate combinations differ between rounding at each')
print(f'step and rounding once at the end ({once/step:.2%})')

Rounding after each step is not cosmetic here. On hospital 1 the two conventions are
indistinguishable, so the development set cannot validate the choice; on hospital 5
they diverge on roughly one rate in nine. That 99.30% of hospital 5 reproduces under
step rounding is the evidence for the reading.

In [ ]:
findings = {}
for h in TABULAR:
    findings[h] = pd.concat([
        tier1[h],
        audit.unknown_service_findings(resolutions[h], lines[h]),
        classify.classify(priced[h], specs[h])],
        ignore_index=True).groupby(['invoice_id', 'category'], as_index=False).agg(
            detail=('detail', 'first'), n_lines=('detail', 'size'))

wide = (pd.concat([f.assign(hospital=h) for h, f in findings.items()])
          .pivot_table(index='category', columns='hospital', values='invoice_id',
                       aggfunc='nunique', fill_value=0))
wide.columns = [c.replace('hospital_', 'H') for c in wide.columns]
wide.loc['TOTAL INVOICES FLAGGED'] = [findings[f'hospital_{c[1:]}'].invoice_id.nunique()
                                      for c in wide.columns]
wide

Hospital 5 shows two categories the others cannot: `wrong_facility_multiplier` and
`wrong_tier_multiplier`. These extend the hospital 1 vocabulary, which has no term for
them because hospital 1 has no multipliers.

Eighteen of thirty-one apparently mispriced hospital 5 lines were the correct service,
correctly adjusted, read from the wrong grid column. Reporting them as
`unit_price_mismatch` would be true but would leave a reviewer to rediscover the
pattern. `unit_price_mismatch` now means the billed rate is not one the contract can
produce at any setting.

---

## 6 · Evaluation on hospital 1

The only hospital with ground truth, and therefore the only place the system can be
measured rather than argued for.

In [ ]:
final = scoring.report(findings[DEV], labels, 'full audit — hospital 1')

In [ ]:
totals = classify.expected_totals(priced[DEV], units[DEV])
check = (totals.sort_values('source_seq').groupby('invoice_id', as_index=False).last()
         .merge(labels[['invoice_id', 'expected_total_cents', 'error_categories']],
                on='invoice_id', suffixes=('_pred', '_true')))
check['exact'] = check.expected_total_cents_pred == check.expected_total_cents_true

print(f'expected_total exact on {check.exact.sum()}/{len(check)} = {check.exact.mean():.3%}')
print()
display(check[~check.exact][['invoice_id', 'error_categories',
                             'expected_total_cents_true', 'expected_total_cents_pred']])

Every category at precision and recall 1.000, and expected totals exact on 909 of 913.

**The four misses are all `daily_cap_exceeded`, and they are not solvable.** The engine
trims the quantity to the contractual cap, which is what the contract entitles the
provider to. The labels restore quantities of 3, 9, 3 and 3 against caps of 4, 12, 12
and 8, with no co-occurring line item to explain the remainder. That pre-inflation
quantity is stated nowhere in the contract, so it is not guessed. All four invoices are
flagged correctly; only their expected total differs, and always conservatively.

A perfect development score is a warning, not a result. Four decisions were calibrated
on these labels — identifier reuse (5 examples), category precedence (2), the
`unknown_service` thresholds (12), and the monetary/compliance split (18). Each is
recorded in `reports/decision_log.md` with the evidence and the risk.

In [ ]:
per_cat = scoring.category_scores(findings[DEV], labels).sort_values('support')
fig, ax = plt.subplots(figsize=(9, 6))
y = np.arange(len(per_cat))
ax.barh(y - .2, per_cat.precision, height=.38, label='precision', color='#3a6ea5')
ax.barh(y + .2, per_cat.recall, height=.38, label='recall', color='#5a9367')
ax.set_yticks(y); ax.set_yticklabels([f'{c}  (n={s})' for c, s in
                                      zip(per_cat.category, per_cat.support)], fontsize=8)
ax.set_xlim(0, 1.05); ax.legend(loc='lower left')
ax.set_title('Hospital 1 — per category, with support')
plt.tight_layout(); plt.show()

### 6.1 · Does it generalise, or has it memorised hospital 1?

A perfect score on the only measurable data is a warning rather than a result. Four
decisions were calibrated on these labels, the categories carry three to twelve
examples each, and hospital 1 is the simplest of the five by construction.

So the score is tested rather than defended. Faults are injected into invoices the
labels mark **clean** — errors the system has never been shown, in invoices it has
never seen fail. Every detection is a genuine generalisation; every flag raised on an
untouched clean invoice is a genuine false positive.

In [ ]:
results_stress = []
for seed in (0, 1, 2):
    u_s, li_s, truth = stress_test.inject(units[DEV], lines[DEV].assign(
        service=lines[DEV].description.map(dict(zip(resolutions[DEV].description,
                                                    resolutions[DEV].service)))),
        specs[DEV], labels, n_per_kind=12, seed=seed)
    res_s = resolver.resolve(specs[DEV], li_s)
    priced_s = pricing.reprice(specs[DEV], u_s, li_s, res_s)
    found_s = pd.concat([checks.run_all(u_s, li_s, headers[DEV]),
                         audit.unknown_service_findings(res_s, li_s),
                         classify.classify(priced_s, specs[DEV])],
                        ignore_index=True)
    stress_table, stress_summary = stress_test.score(found_s, truth, labels)
    results_stress.append({'seed': seed, **stress_summary})
    if seed == 0:
        display(stress_table)
pd.DataFrame(results_stress)

**96 injected faults, 96 detected, 96 named with the correct category, on every seed.**
Against 759 clean control invoices, **zero false positives**.

These are errors that appear nowhere in the label file, in invoices the labels call
correct. The system had no opportunity to memorise them, so this measures the engine
rather than the calibration.

It does not clear the four fitted decisions. Injection covers the categories that can
be synthesised unambiguously — arithmetic, dates, rates, units, caps — and not the
ones needing a judgement about intent, such as which invoice of a reused-identifier
pair is the offender. Those remain calibrated on five and two examples respectively
and are the weakest claims in the submission.

The honest summary: **the engine generalises; the tie-breaking conventions are
asserted.** Both belong in the report, separately.

---

## 7 · Model selection

Hospital 2 is the only contract requiring a model. Four candidates, one task, marked
against a gold specification that cost nothing to produce.

| | weights | runs on | data leaves the machine |
|---|---|---|---|
| GPT-4o | closed | vendor | yes |
| Kimi K2 | open | vendor | yes |
| DeepSeek V4.1-Flash | open, MIT | vendor | yes |
| Qwen2.5-7B-Instruct | open | **this notebook** | **no** |

The last row is the one that matters for a payer handling patient data. The accuracy
cost of keeping inference in-house is measured below, not assumed.

Requires `OPENROUTER_API_KEY` in Colab Secrets and a GPU runtime.

In [ ]:
PROMPT_PROSE = open(f'{PROJECT}/prompts/contract_extraction_v1.txt').read()
TABLE_PROMPT_VERSION = 'v3'   # flat scalars: v2's nested list cost the local model 40% of rows
PROMPT_TABLE = open(f'{PROJECT}/prompts/contract_extraction_table_{TABLE_PROMPT_VERSION}.txt').read()
# Every section that states a rate. 'Additional Services' was missing, so the two
# services the amendment adds were shown to no model and counted against all four.
TABLE_SECTIONS = ['Rate Schedule', 'Base Rates', 'Table 1', 'B.1',
                  'Substituted Rates', 'Additional Services']
EXAM = ['hospital_3', 'hospital_4', 'hospital_5']

def exam_chunks(hospital):
    text = ''.join(open(f).read() for f in sorted(glob.glob(f'{DATA}/contracts/{hospital}/*.md')))
    return chunking.table_chunks(text, TABLE_SECTIONS)

HAVE_KEY = llm.get_key('OPENROUTER_API_KEY') is not None
print(f'api key {HAVE_KEY}   gpu {HAS_GPU}')
for h in EXAM:
    c = exam_chunks(h)
    print(f'{h}: {len(c):2d} batches, {sum(x["n_clauses"] for x in c):3d} rows, '
          f'largest {max(x["n_clauses"] for x in c)}')

Rate tables are batched to fifteen rows. Output length, not context length, is the
binding constraint: hospital 3's schedule is 118 rows and would need roughly 9,400
output tokens against a 4,096 limit, truncating every reply mid-object.

Two prompts, because these are different reading tasks — numbered prose clauses for
hospital 2, rate tables for the exam. Using one on the other would measure format
mismatch rather than extraction quality.

### 7.1 · Recorded runs

Every raw reply is written to `runs/` as each contract completes, and replayed on a
later run instead of being requested again. Inference is the only non-deterministic,
non-free step in this notebook; recording it makes the submission reproducible by
anyone who clones the repository, with no credential, no GPU and no spend.

Set `FORCE_RERUN = True` to ignore the recordings and call the models again.

On Colab, `runs/` is cleared when the session ends. If `RUNS_BACKUP` points at a
Google Drive folder, recordings are mirrored there and restored automatically.

In [ ]:
FORCE_RERUN = False
RUNS_BACKUP = '/content/drive/MyDrive/invoice-audit-runs' if IN_COLAB else None
RUNS = f'{PROJECT}/runs'


def mount_backup():
    if not (IN_COLAB and RUNS_BACKUP and drive):
        return 0
    drive.mount('/content/drive', force_remount=False)
    os.makedirs(RUNS_BACKUP, exist_ok=True)
    restored = 0
    for name in os.listdir(RUNS_BACKUP):
        if not name.endswith('.json'):
            continue
        if not os.path.exists(f'{RUNS}/{name}'):
            shutil.copy(f'{RUNS_BACKUP}/{name}', f'{RUNS}/{name}')
            restored += 1
    return restored


def save_backup():
    if not (IN_COLAB and RUNS_BACKUP and os.path.isdir(RUNS_BACKUP)):
        return 0
    # Only the recordings. runs/ also holds working directories, and
    # shutil.copy raises on a directory.
    saved = 0
    for name in os.listdir(RUNS):
        if name.endswith('.json'):
            shutil.copy(f'{RUNS}/{name}', f'{RUNS_BACKUP}/{name}')
            saved += 1
    return saved


def run_path(model_name, hospital, version):
    """Recordings are keyed by prompt version as well as model and hospital."""
    return f'{RUNS}/{model_name}__{version}__{hospital}.json'


def recorded(model_name, hospital, version, require_usage=True):
    """A usable recording, or None.

    A recording written before usage was captured holds only the reply text. It
    replays the right answers but reports no tokens and no latency, which would
    leave blank rows in a comparison whose whole purpose is to compare cost and
    speed. Such a recording is treated as absent so the model runs again, rather
    than being silently accepted and quietly reported as free.
    """
    path = run_path(model_name, hospital, version)
    if not os.path.exists(path):
        return None
    rec = json.load(open(path))
    if require_usage:
        first = rec.get('0')
        if not (isinstance(first, dict) and 'usage' in first):
            return None
    return rec


def recording_status(version):
    """What is on disk for this prompt version, and whether it carries telemetry."""
    rows = []
    for name in sorted(os.listdir(RUNS)):
        if not name.endswith('.json') or f'__{version}__' not in name:
            continue
        rec = json.load(open(f'{RUNS}/{name}'))
        first = rec.get('0')
        rows.append({'recording': name,
                     'chunks': len(rec),
                     'telemetry': 'yes' if isinstance(first, dict) and 'usage' in first
                                  else 'no — will re-run'})
    return pd.DataFrame(rows)


def run_model(model_name, hospitals, prompt, version, verbose=True):
    """Extract each contract, replaying only recordings that carry telemetry."""
    out, model = {}, None
    for h in hospitals:
        replay = None if FORCE_RERUN else recorded(model_name, h, version)
        if replay is not None:
            source = llm.ReplayModel(replay, model_name)
        else:
            if model is None:
                schema = local_model.EXTRACTION_SCHEMA if version == 'v3' else None
                model = (local_model.LocalModel(schema=schema)
                         if model_name.startswith('qwen')
                         else llm.ApiModel(model_name, schema=schema))
            source = model
        record = {}
        if verbose:
            print(f"--- {model_name} · {version} · {h} · "
                  f"{'replayed' if replay is not None else 'live'} ---")
        spec, frame, tel = extract.extract_contract(
            source, prompt, exam_chunks(h), headers[h], h,
            verbose=verbose, record=record)
        if replay is None:
            json.dump(record, open(run_path(model_name, h, version), 'w'), indent=1)
            save_backup()
        out[h] = {'spec': spec, 'frame': frame, 'telemetry': tel,
                  'replayed': replay is not None}
    return out


print(f'restored from backup: {mount_backup()} files')
display(recording_status(TABLE_PROMPT_VERSION))

In [ ]:
def smoke_test(name='gpt-4o'):
    """One live call, before committing to the sweep."""
    model = llm.ApiModel(name)
    chunk = exam_chunks('hospital_5')[0]
    reply, usage = model.generate(PROMPT_TABLE, chunk['text'])
    payload, err = llm.parse_json(reply)
    print(f'{model.model_id} via {model.via}  {usage}')
    print(f'parsed: {err or "ok"}')
    if payload:
        got = payload.get('services', [])
        print(f'returned {len(got)} of {chunk["n_clauses"]} rows')
        if got:
            print(got[0])
    return payload


if HAVE_KEY and not any(f.endswith('.json') for f in os.listdir(RUNS)):
    _ = smoke_test()
else:
    print('skipped: recordings present, or no credential')

One call before spending on the sweep. It confirms the credential works, the reply
parses, and rates return as integers rather than decimals.

In [ ]:
# A provider outage on one model must not discard the others. Each is attempted
# independently and its failure recorded rather than raised.
results, failed = {}, {}

if HAVE_KEY or any(f.endswith('.json') for f in os.listdir(RUNS)):
    for name in ['gpt-4o', 'kimi-k2', 'deepseek']:
        try:
            results[name] = run_model(name, EXAM, PROMPT_TABLE, TABLE_PROMPT_VERSION)
        except Exception as exc:
            failed[name] = f'{type(exc).__name__}: {exc}'
            print(f'{name}: {failed[name][:160]}')

print(f'\ncompleted {sorted(results)}   failed {sorted(failed)}')
print(f'backed up {save_backup()} recordings')

In [ ]:
LOCAL_TAG = 'qwen-4bit' if _bnb_ok else 'qwen-fp16'

if HAS_GPU or recorded(LOCAL_TAG, 'hospital_3', TABLE_PROMPT_VERSION):
    results[LOCAL_TAG] = run_model(LOCAL_TAG, EXAM, PROMPT_TABLE, TABLE_PROMPT_VERSION)

print(f'\nbacked up {save_backup()} recordings to Drive')

In [ ]:
rows = []
for name, per_hospital in results.items():
    for h, res in per_hospital.items():
        # Hospital 4 states caps outside its rate table, so caps are scored only where
        # the table the model saw actually carries them.
        fields = {'rate', 'unit_basis'} | ({'daily_cap'} if h != 'hospital_4' else set())
        m = spec_diff.compare_specs(specs[h], res['spec'], score_fields=fields)
        t = model_comparison.telemetry_summary(res['telemetry'], name)
        rows.append({'model': name, 'hospital': h,
                     'found': f"{m['found_services']}/{m['gold_services']}",
                     'rate_exact': m['rate_exact'],
                     'basis_exact': m['unit_basis_exact'],
                     'cap_exact': m['daily_cap_exact'],
                     'hallucinated': m['hallucinated_services'] + m['hallucinated_rules'],
                     'parse_fail': t['parse_failures'],
                     'schema_dev': t['schema_deviations'],
                     'quote_unverified': t['quotes_unverified'],
                     'schema_enforced': t['schema_enforced'],
                     'tokens_in': t['input_tokens'], 'tokens_out': t['output_tokens'],
                     'seconds': t['seconds'],
                     'telemetry': 'recorded' if res.get('replayed') else 'live'})
scores = pd.DataFrame(rows)
scores if len(scores) else print('run the cells above first')

In [ ]:
overall = None

if len(scores):
    overall = (scores.groupby('model').agg(
        rate_exact=('rate_exact', 'mean'), basis_exact=('basis_exact', 'mean'),
        hallucinated=('hallucinated', 'sum'), parse_fail=('parse_fail', 'sum'),
        schema_dev=('schema_dev', 'sum'),
        tokens_in=('tokens_in', 'sum'), tokens_out=('tokens_out', 'sum'),
        seconds=('seconds', 'sum')).round(3).sort_values('rate_exact', ascending=False))
    display(overall)

    fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
    overall.rate_exact.plot(kind='barh', ax=ax[0], color='#3a6ea5')
    ax[0].set_title('Rate exact-match'); ax[0].set_xlim(0, 1)
    overall.hallucinated.plot(kind='barh', ax=ax[1], color='#b03a3a')
    ax[1].set_title('Rules invented (lower better)')
    overall.seconds.plot(kind='barh', ax=ax[2], color='#7a7a7a')
    ax[2].set_title('Wall-clock seconds')
    plt.tight_layout(); plt.show()
else:
    print('no model results yet — run the sweep above')

Invented rules are plotted separately rather than folded into an accuracy figure.
A missed rule leaves a gap the engine reports; an invented one is a confident value
with no source that misprices every invoice touching that service.

`schema_enforced` marks where the output shape was **guaranteed** rather than
requested. The hosted models declare the schema to the API, which constrains their
sampler; the local model is constrained here in the notebook, which is only
possible because it runs here — a remote sampler cannot be constrained by the
caller. Where enforcement was unavailable the run falls back to prompting and the
column says so, because a model that merely *usually* returns the right shape is a
different operational proposition from one that cannot return a wrong one.

`schema_dev` and `quote_unverified` should both fall to zero wherever
`schema_enforced` is true. That is the check on whether enforcement is doing what
it claims.

In [ ]:
if len(scores):
    for name, per_hospital in results.items():
        d = spec_diff.diff_report(specs['hospital_5'], per_hospital['hospital_5']['spec'], limit=3)
        print(f'--- {name} · hospital 5 · {len(d)} differences ---')
        print(d.head(6).to_string(index=False) if len(d) else '   none')
        print()

### 7.2 · Did the prompt revision help?

v1 specified a single `rate_cents`. Hospital 3's amended Appendix B prices seven
services across two dated columns, which that schema cannot represent, so v1's ceiling
on this exam was `(113/120 + 1 + 1) / 3 = 0.980` — exactly what three of the four
models scored. They were at the maximum the prompt allowed; the shortfall was the
schema's.

v2 makes `rates` a list carrying `valid_from` and `valid_to`. That introduces a risk
worth measuring rather than assuming away: a list invites a model to emit two rates
where the contract states one. **Hospitals 4 and 5 have no dated rates, so their score
must stay at 1.000.** A drop there means the schema taught the model to invent
periods, and the revision is a net loss.

Recordings are keyed by prompt version, so both runs survive and can be compared.

In [ ]:
versions = {}
for _version in ('v1', 'v2'):
    _prompt_path = f'{PROJECT}/prompts/contract_extraction_table_{_version}.txt'
    if not os.path.exists(_prompt_path):
        continue
    _prompt = open(_prompt_path).read()
    for _name in sorted(results):
        if not all(recorded(_name, h, _version) for h in EXAM):
            continue
        _run = run_model(_name, EXAM, _prompt, _version, verbose=False)
        for h, res in _run.items():
            versions[(_version, _name, h)] = spec_diff.compare_specs(
                specs[h], res['spec'])['rate_exact']

if versions:
    compare = (pd.Series(versions).rename('rate_exact').reset_index()
                 .rename(columns={'level_0': 'prompt', 'level_1': 'model',
                                  'level_2': 'hospital'})
                 .pivot_table(index=['model', 'hospital'], columns='prompt',
                              values='rate_exact'))
    if {'v1', 'v2'} <= set(compare.columns):
        compare['delta'] = (compare['v2'] - compare['v1']).round(3)
    display(compare.round(3))
else:
    print('no completed run for either prompt version yet')

Read the `hospital_3` rows for whether the revision worked, and `hospital_4` and
`hospital_5` for whether it cost anything. A gain on 3 with 4 and 5 unchanged is the
result the revision was made to produce. A gain on 3 paid for by a loss elsewhere
would mean the schema is now encouraging invented periods.

Recorded in `prompts/CHANGELOG.md`.

### 7.4 · Where a model loses rows

A shortfall has two possible causes with different remedies: the model returned fewer
objects than it was given rows, or it returned enough and validation rejected some. The
first is a generation problem, the second a schema-conformance one.

In [ ]:
# Where a model loses rows: did it return too few, or return enough and fail validation?
for _name, _per in results.items():
    _tel = pd.concat([r['telemetry'] for r in _per.values()])
    _short = _tel[_tel.services_accepted < _tel.clauses_expected]
    if not len(_short):
        print(f'{_name:12s} no shortfall')
        continue
    _returned = int(_short.services_returned.sum())
    _expected = int(_short.clauses_expected.sum())
    _accepted = int(_short.services_accepted.sum())
    print(f'{_name:12s} on {len(_short)} short batches: expected {_expected}, '
          f'returned {_returned}, accepted {_accepted}')
    print(f'{"":12s}   -> {_expected - _returned} never returned, '
          f'{_returned - _accepted} returned but rejected')

# and the reasons, for the worst offender
_worst = max(results, key=lambda n: sum(
    (r['telemetry'].clauses_expected - r['telemetry'].services_accepted).sum()
    for r in results[n].values()))
print()
print(f'rejection reasons for {_worst}:')
for _w in results[_worst]['hospital_3']['spec']['warnings'][:12]:
    print('   ', _w[:120])

### 7.3 · Retrieval, measured rather than assumed

Section 3 selected rate-bearing articles by heading and dropped 49% of hospital 2. The
alternative is an embedding index. The claim was that top-*k* retrieval cannot
guarantee completeness when every rate clause is required; this measures it.

In [ ]:
h2_chunks = chunking.rate_chunks(h2_text) + [
    {'title': c['title'], 'text': c['text'], 'n_clauses': 0}
    for c in chunking.convention_chunks(h2_text)]
all_articles = [{'title': t, 'text': b.strip()} for t, b in markdown_tables.sections(h2_text).items()]

index = retrieval.VectorIndex([a['text'] for a in all_articles],
                               [{'title': a['title'], 'hospital': PROSE} for a in all_articles],
                               encoder)
wanted = {c['title'] for c in chunking.rate_chunks(h2_text)}

print(f'{len(all_articles)} articles indexed, {len(wanted)} of them carry rates')
print(f'{"k":>4}  {"rate articles in top-k":>24}  {"recall":>7}')
for k in (5, 10, 13, 20, 26):
    hits = index.search('rate per unit payable for this service', k=k, where={'hospital': PROSE})
    found = len({h['title'] for h in hits} & wanted)
    print(f'{k:>4}  {found:>24}  {found/len(wanted):>7.1%}')
print()
print('structural filter on the heading: 13 of 13, 100.0%, no query needed')

The ranking is good: all thirteen rate articles sort above all twenty-two
boilerplate ones, so at *k*=13 recall is perfect. Retrieval is not failing here.

It is the *k* that is the problem. At *k*=10 the index silently drops three rate
articles and 23% of the contract's rates with them, and nothing in the scores says
so — the tenth and fourteenth results look equally plausible. Choosing *k*=13
requires already knowing the answer, which is precisely what the heading filter
tells you for free, exactly, with no query and no model.

So the decision is not that embeddings are worse at ranking. It is that a task
requiring *every* member of a known set should use the property that defines the
set, not a similarity score that approximates it.

The index is kept for the case this project does not have — a corpus too large to
read exhaustively — and because its metadata filter demonstrates the constraint
that matters at five contracts: a query scoped to one hospital must never return
another's clause. Similarity is a score; a hospital identifier is a fact.

---

## 8 · Ask the contract

The extraction above runs in batch. This is the same machinery, interactive: a
question goes in, clauses are retrieved from **one hospital's** contract, and the model
answers from those clauses and nothing else.

The scoping is the point. Retrieval is filtered on `hospital` **before** similarity is
computed, so a question about hospital 2 cannot be answered with hospital 4's rates —
similarity is a score, a hospital identifier is a fact. Every answer carries the clause
it came from, so it can be checked against the source in seconds.

In [ ]:
clause_indexes = {}
for h in HOSPITALS:
    clause_indexes[h], _clauses = ask_mod.clause_index(DATA, h, encoder)
    print(f'{h}: {len(_clauses)} clauses indexed')

In [ ]:
ASK_MODEL = llm.ApiModel('gpt-4o') if HAVE_KEY else None

def ask(question, hospital=PROSE, k=4):
    """Ask a question about one hospital's contract."""
    if ASK_MODEL is None:
        hits = clause_indexes[hospital].search(question, k=k, where={'hospital': hospital})
        print(f'no API key — showing retrieval only for {hospital}')
        for i, h in enumerate(hits):
            print(f'   [{i+1}] {h["score"]:.3f}  {h["section"][:64]}')
        return
    return ask_mod.ask(question, clause_indexes[hospital], ASK_MODEL, hospital, k=k)

ask('What is the rate for Assisted Infectious Telemetry Monitoring, and on what basis?')

In [ ]:
ask('How does this contract define a Business Day, and does it differ from a calendar day?')

In [ ]:
ask('Which services carry a cumulative volume discount, and at what thresholds?')

### Testing whether it refuses

The useful question is not whether it answers correctly when the clause is present —
it is whether it declines when the clause is absent. A model that invents a plausible
rate is worse than one that says it cannot find it, because an invented rate propagates
silently into every invoice touching that service.

In [ ]:
ask('What is the rate for Advanced Cardiac Ventilation Support?', hospital=PROSE)
# that service belongs to hospital 5, not hospital 2 — the answer should be a refusal

In [ ]:
ask('What penalty applies if the Provider submits an invoice more than 90 days late?')
# no such clause exists in any contract

> A contract question answered from retrieved clauses is the same capability the batch
> extraction uses, exposed one question at a time. It is also how the extraction was
> debugged: when a rate looked wrong, asking the contract directly was faster than
> re-reading 35 articles.

---

## 8 · Hospital 2

The winning model reads all thirteen rate articles. There is no gold specification
here — that is why a model is needed — so the extraction is checked by whether the
resulting rates reproduce the amounts the hospital actually billed.

A correct specification reproduces 99.3–99.5% of line items on the four hospitals
where it can be verified. A materially lower figure means the contract was misread,
and that is knowable without any label.

In [ ]:
# Selection rule, stated before the numbers are looked at: among models that
# invented nothing and returned parseable JSON every time, take the fastest.
# Accuracy did not separate them, so latency is the remaining axis that matters
# for a contract that must be re-read whenever it is amended.
PROSE_PROMPT_VERSION = 'prose_v1'

if overall is not None and len(overall):
    clean = overall[(overall.hallucinated == 0) & (overall.parse_fail == 0)]
    BEST = (clean.sort_values('seconds').index[0] if len(clean)
            else overall.index[0])
    print(f'models with no hallucinations and no parse failures: {list(clean.index)}')
    print(f'selected: {BEST}  ({overall.loc[BEST, "seconds"]:.0f}s on the exam)')
else:
    BEST = 'gpt-4o'
    print(f'no comparison available; defaulting to {BEST}')

h2_articles = chunking.rate_chunks(h2_text)
print(f'hospital 2: {len(h2_articles)} articles, '
      f'{sum(a["n_clauses"] for a in h2_articles)} clauses to extract')

h2_replay = recorded(BEST, PROSE, PROSE_PROMPT_VERSION)
if h2_replay is not None:
    h2_source = llm.ReplayModel(h2_replay, BEST)
    print('replaying a recorded run')
else:
    h2_source = (local_model.LocalModel(schema=local_model.EXTRACTION_SCHEMA)
                 if BEST.startswith('qwen')
                 else llm.ApiModel(BEST))
    print('running live')

h2_record = {}
spec_h2, frame_h2, tel_h2 = extract.extract_contract(
    h2_source, PROMPT_PROSE, h2_articles, headers[PROSE], PROSE, record=h2_record)

if h2_replay is None:
    json.dump(h2_record, open(run_path(BEST, PROSE, PROSE_PROMPT_VERSION), 'w'), indent=1)
    save_backup()

print()
print(contracts.summarise(spec_h2))

In [ ]:
expected_clauses = sum(a['n_clauses'] for a in h2_articles)
print(f'clauses in the contract  {expected_clauses}')
print(f'services extracted       {len(spec_h2["services"])}')
print(f'rules: {sum(len(v) for v in spec_h2["volume_discounts"].values())} discounts · '
      f'{len(spec_h2["threshold_premiums"])} premiums · '
      f'{len(spec_h2["nbd_uplifts"])} uplifts · {len(spec_h2["bundles"])} bundles')
if spec_h2['warnings']:
    print()
    print(f'{len(spec_h2["warnings"])} rejected or unresolved:')
    for w in spec_h2['warnings'][:8]:
        print('   ', w[:120])
display(frame_h2.head(8))

In [ ]:
specs[PROSE] = spec_h2
resolutions[PROSE] = resolver.resolve(spec_h2, lines[PROSE])
priced[PROSE] = pricing.reprice(spec_h2, units[PROSE], lines[PROSE],
                                     resolutions[PROSE])

# Hospital 2 joins the same normalised table as the other four. Its rows are
# marked as model-extracted, so a reviewer can see which rates were read by a
# model and which by regex.
contracts_table = contract_table.combine(contract_tables + [
    contract_table.spec_to_table(spec_h2, 'model_extracted', 0.90)])
contracts_table.to_csv(f'{PROJECT}/reports/contracts_normalised.csv', index=False)

_p2 = priced[PROSE]
_ok = _p2.expected_line_total.notna()
_match = (_p2.expected_line_total == _p2.line_total_cents)
print(f'line items priced        {int(_ok.sum()):,}/{len(_p2):,}')
print(f'reproduce billed amount  {_match.sum()/_ok.sum():.2%}    '
      f'(regex-read hospitals: 99.30-99.51%)')
print(f'descriptions unresolved  {int((resolutions[PROSE].method == "unresolved").sum())}')
print()
display(contract_table.summarise(contracts_table))

The reproduction rate is the verdict on the extraction. If it lands in the same band
as the four verified hospitals, the model read the contract correctly; if not, the
specification is wrong and the affected services carry reduced confidence.

This also settles the Service Day ambiguity from section 3. A wrong reading would show
up only on the eight services carrying a non-business-day uplift, and only on dates
adjacent to weekends.

In [ ]:
if PROSE in priced:
    nbd_services = set(spec_h2['nbd_uplifts'])
    p2 = priced[PROSE]
    affected = p2[p2.service.isin(nbd_services) & p2.expected_line_total.notna()]
    weekend = affected.service_date.dt.weekday >= 5
    for label, subset in [('non-business-day services', affected),
                          ('  of those, weekend dates', affected[weekend]),
                          ('  of those, weekday dates', affected[~weekend])]:
        if len(subset):
            agree = (subset.expected_line_total == subset.line_total_cents).mean()
            print(f'{label:28s} {len(subset):5d} lines, {agree:.2%} reproduce')

Equal reproduction rates on weekend and weekday dates means the reading is consistent
with the data. A systematic shortfall confined to weekends would indicate the 07:00
Service Day boundary is operative and the alternative reading is required.

In [ ]:
findings[PROSE] = pd.concat([
    tier1[PROSE],
    audit.unknown_service_findings(resolutions[PROSE], lines[PROSE]),
    classify.classify(priced[PROSE], spec_h2)],
    ignore_index=True).groupby(['invoice_id', 'category'], as_index=False).agg(
        detail=('detail', 'first'), n_lines=('detail', 'size'))

_n, _total = findings[PROSE].invoice_id.nunique(), units[PROSE].invoice_id.nunique()
print(f'hospital 2 flagged {_n}/{_total} = {_n/_total:.1%}'
      f'   (hospital 1 true rate 6.4%)')
display(findings[PROSE].category.value_counts().to_frame('invoices'))

---

## 9 · Submission

A row for every invoice in hospitals 2–5, including those believed correct, since the
task states those are scored.

Confidence is composed rather than asserted: it is bounded by the weakest evidence
beneath each finding — how the description was resolved, whether the contract was read
by regex or by a model, and whether the category is an arithmetic fact or an
interpretation.

In [ ]:
CATEGORY_CONFIDENCE = {
    'line_total_arithmetic': 1.00, 'invoice_total_mismatch': 1.00,
    'duplicate_invoice_id': 1.00, 'malformed_service_date': 1.00,
    'service_date_after_invoice_date': 1.00, 'service_date_out_of_window': 1.00,
    'contract_number_mismatch': 1.00,
    'unknown_service': 0.90, 'wrong_unit_basis': 0.92, 'unit_price_mismatch': 0.88,
    'cross_invoice_duplicate': 0.90, 'exclusion_window_violation': 0.85,
    'bundle_not_applied': 0.88, 'premium_omitted': 0.85,
    'premium_incorrectly_applied': 0.85, 'volume_discount_omitted': 0.85,
    'volume_discount_incorrectly_applied': 0.85, 'daily_cap_exceeded': 0.75,
    'wrong_facility_multiplier': 0.88, 'wrong_tier_multiplier': 0.88,
}
EXTRACTION_PENALTY = {'regex': 1.00, 'model': 0.85}


def invoice_confidence(hospital, invoice_id, categories):
    base = min(CATEGORY_CONFIDENCE.get(c, 0.7) for c in categories)
    source = 'model' if hospital == PROSE else 'regex'
    if all(CATEGORY_CONFIDENCE.get(c, 0) == 1.0 for c in categories):
        return round(base, 3)
    return round(base * EXTRACTION_PENALTY[source], 3)

print('confidence = weakest category evidence x extraction-source penalty')
print('arithmetic facts are exempt from the penalty: they hold whoever read the contract')

In [ ]:
rows = []
for h in SCORED:
    if h not in findings:
        print(f'{h}: skipped, no findings'); continue
    tot = classify.expected_totals(priced[h], units[h])
    tot = tot.sort_values('source_seq').groupby('invoice_id', as_index=False).last()
    by_invoice = findings[h].groupby('invoice_id').category.apply(list).to_dict()
    for r in tot.itertuples():
        cats = by_invoice.get(r.invoice_id, [])
        rows.append({
            'invoice_id': r.invoice_id,
            'flagged': int(bool(cats)),
            'error_category': '|'.join(sorted(cats)),
            'expected_total_cents': int(r.expected_total_cents),
            'billed_total_cents': int(r.invoice_total_cents),
            'confidence': invoice_confidence(h, r.invoice_id, cats) if cats else 0.95,
        })
submission = pd.DataFrame(rows)
print(f'{len(submission)} rows, {submission.flagged.sum()} flagged '
      f'({submission.flagged.mean():.1%})')
submission.head()

In [ ]:
assert submission.invoice_id.is_unique, 'duplicate invoice_id'
assert submission.flagged.isin([0, 1]).all(), 'flagged not 0/1'
assert submission.confidence.between(0, 1).all(), 'confidence out of range'
assert submission.expected_total_cents.map(lambda v: float(v).is_integer()).all()
assert submission.billed_total_cents.map(lambda v: float(v).is_integer()).all()
assert (submission.loc[submission.flagged == 1, 'error_category'] != '').all()
assert list(submission.columns) == open(f'{DATA}/submission_template.csv').read().strip().split(',')

submission.to_csv(f'{PROJECT}/submission.csv', index=False)
print(f'submission.csv written — {len(submission)} rows')
print()
display(submission.groupby('flagged').confidence.describe()[['count', 'mean', 'min', 'max']])

---

## Summary

| | |
|---|---|
| Hospital 1 detection | precision 1.000, recall 1.000, F1 1.000 |
| Hospital 1 expected totals | exact on 909 of 913 |
| Line items reproduced | 99.30–99.51% across four verified hospitals |
| Descriptions unresolved | none |
| Model calls for the whole submission | under 150 |

Written up in `reports/evaluation.md`; assumptions and unresolved ambiguities in
`reports/decision_log.md`; prompts and their revisions in `prompts/`; every raw model
reply in `runs/`, so this notebook reproduces the submission without a credential.